# Cancel a running workflow

Run `qdash config init --project-id your-project-id` in a terminal, then fill in `api_token` in the generated file. Use `--base-url` for a remote deployment. If the file already exists, edit it instead. `QDash()` reads its `default_profile`; use `QDash.from_profile("local")` to select a profile explicitly. See the [Python reference](../../../docs/reference/clients/python.md#connect-to-a-deployment) for setup. The workflow must already be saved and deployed in QDash, with its worker available. Cancellation requires editor access to the configured project and a worker able to process it. Run these cells one at a time.

In [ ]:
from qdash.sdk import QDash, QDashConflictError, QDashTransportError

qdash = QDash()

## Select a workflow

Choose an existing deployment that is suitable to start and stop on your hardware. Review its chip and parameters before launching.

In [ ]:
for item in qdash.workflows.list():
    print(item.name, item.chip_id)

workflow = qdash.workflows.get("YOUR_WORKFLOW_NAME")
print("Chip:", workflow.chip_id)
print("Defaults:", workflow.default_parameters)
parameters = {}

## Start the workflow

This cell starts real work. Run it once and retain `run`; re-running it launches another workflow. If the response is lost, inspect QDash before submitting again.

In [ ]:
run = workflow.run(parameters=parameters)
print("Execution:", run.execution_id)
print("Flow run:", run.flow_run_id)

## Check progress

Repeat this read-only cell until you want to cancel. Interrupting a notebook cell or timing out a wait does not cancel the remote workflow.

In [ ]:
state = run.get()
print(state.status, "Done:", state.done)

## Request cancellation

This cancels the whole workflow, including its steps. `cancelling` acknowledges the request; it is not a terminal result. If the run finishes first, its actual terminal state remains. After a connection error, check status before deciding whether to retry.

In [ ]:
state = run.get()
if state.done:
    print("Already finished:", state.status)
else:
    try:
        cancellation = run.cancel()
        print(cancellation.status, cancellation.message)
    except QDashConflictError:
        print("Cancellation was refused; current state:", run.get().status)
    except QDashTransportError:
        print("Cancellation response was lost. Check progress before retrying.")
        raise

## Confirm the final state

Wait for the whole workflow to end. If this times out, repeat this cell or check progress; do not re-run the launch cell. If it remains `cancelling`, check the worker in QDash. A final state other than `cancelled` means cancellation did not determine the outcome.

In [ ]:
result = run.wait(timeout_seconds=600)
print("Final state:", result.status)
print("Cancelled:", result.status == "cancelled")